# [ISR06 미션 08] AAR을 통한 보고 불일치 해결 실패 원인 분석 및 교차검증 루브릭 개선 — 교수자용 해설판 (Workbook)

- **산출물 경로**: `reports/isr06_redteam_critique_report.md`
- **실습 환경**: Python 3.12 표준 라이브러리 (`pathlib`, `csv`) — 별도 라이브러리 설치 및 인터넷 연결 불필요

> **실습·보안 안내**: 제공된 교육용 가상자료만 사용하며 실제 조직명·위치·수집 수단·개인정보는 입력하지 마세요. 점수나 등급으로 승인을 자동 결정하지 않으며, 최종 판단은 반드시 사람이 내립니다.

---

## 미션 8. 문제가 생긴 원인과 문서 개선안 (선택·가점/참고)

### 목표

이번 미션에서는 교육장에서 제공하는 생성형 AI와 함께 「문제가 생긴 원인과 문서 개선안」 초안을 만듭니다.

결과가 좋지 않았다는 말만으로는 다음 문서를 고칠 수 없습니다.

어떤 자료와 절차가 영향을 주었는지 시간순으로 정리하고 수정할 문장까지 연결합니다.

준비할 자료는 다음과 같습니다: 서로 독립된 업무 일지 A·B, 미션 2·4·5 기준. AI가 제안한 초안을 받은 뒤 사실과 숫자, 빠진 항목, 과장된 표현을 확인하고 직접 고쳐 씁니다.

끝나면 완성한 문서와 자료 대조 기록, 사용한 프롬프트, 고친 이유가 남습니다.

문장마다 근거를 남겨 다음 미션에서도 같은 자료를 찾아 쓸 수 있게 합니다.

### 상황과 자료

가상 업무 일지 A·B에는 계획과 다른 결과가 남아 있습니다.

A팀은 개인을 비난하지 않고 기록에 근거해 개선할 조항을 찾습니다.

**준비할 자료**: 서로 독립된 업무 일지 A·B, 미션 2·4·5 기준

**뜻 알아두기**: AAR은 일을 마친 뒤 계획과 실제를 비교하는 돌아보기입니다.

레드팀은 초안의 허점을 찾는 검토자입니다.

두 일지의 시각을 합치지 않습니다.

반박을 모두 채택하지 않아도 근거와 이유가 있으면 됩니다.

이 미션은 선택입니다.

In [ ]:
# [환경 준비] 미션 8 실습 데이터셋 및 빈 산출물 서식 생성 (기존 파일은 보존)
from pathlib import Path

payload = {'02_데이터셋/출처_이력대장.csv': "[교육용가상자료] 출처 이력대장 — 가상 조직 A팀 출처 풀. 등록일·보고 건수는 D일 기준이며 실제 수집 수단·인물과 무관하다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n출처ID,출처유형,출처설명,등록일,과거보고건수,확인된적중건수,적중률_퍼센트,검증이력상태,전달경로,평균전달지연,비고\nH-1,인터뷰,현지 협조자(나-3 교량 북쪽 마을 거주),D-210일,8,6,87.5,검증 이력 있음,연락관 경유 구두 보고,약 6시간,관측 시각을 보고에 적지 않는 습관이 있음\nH-2,인터뷰,신규 협조자(H-1의 소개),D-1일,0,0,해당 없음,검증 이력 없음,H-1 경유 전언,불명,첫 보고가 H-1에게 들은 내용의 전달\nH-3,인터뷰,다-2 도로변 협조자,D-95일,3,2,66.7,검증 이력 있음,직접 통화,약 40분,시각을 '○○시경'으로만 적음\nS-1,통신기록,가상 통신 요약 판독팀,상시 운용,40,36,90.0,검증 이력 있음,자동 요약 후 판독관 확인,약 1시간,통신 내용은 포함하지 않고 통신량·구역만 요약\nI-1,사진자료,위성영상 판독팀,상시 운용,25,23,92.0,검증 이력 있음,판독 의견서,약 20분,영상 시각·해상도 등급 기재. 구름·야간에 공백\nI-2,사진자료,드론 영상 운용조(다-2 도로 전담),상시 운용,12,11,91.7,검증 이력 있음,실시간 전송,약 5분,배터리 교체마다 20분 공백\nO-1,공개자료,공개 게시 계정(신원 불명),등록 없음,0,0,해당 없음,검증 불가,공개망 열람,해당 없음,이전 게시물 사진 재사용 전력 1회 의심\nO-2,공개자료,공개 게시 계정(O-1 인용 계정군),등록 없음,0,0,해당 없음,검증 불가,공개망 열람,해당 없음,O-1 사진을 인용해 게시\n", '02_데이터셋/보고_접수대장.csv': "[교육용가상자료] 보고 접수대장 — 가상 조직 A팀 D일 접수분(사전 접수분 포함). '(미기재)'는 보고에 그 값이 없었다는 뜻이며 추정해 채우지 않는다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n보고ID,출처ID,출처유형,관측시각,접수시각,격자,내용요약,재인용원출처,비고,쓰이는미션\nP2,H-2,인터뷰,(미기재),D-3일 09:10,가-4,가-4 일대 낯선 차량 2대 목격,,사전 접수분,참고\nP1,H-1,인터뷰,D-2일 14:00,D-2일 20:30,나-3,교량 인근 차량 통행 평시 수준,,사전 접수분,참고\nC1,H-1,인터뷰,(미기재),07:40,나-3 북단,군용 차량 10여 대 정차,,연락관 추정 전달 지연 약 6시간,M1·M2(R1)·M3(K1-A)·M5\nC2,S-1,통신기록,05:00~07:00(수집 구간),07:55,가-4,통신량 평소의 3배,,통신 내용 불명,M1·M2(R4)·M3(K4-A)\nC5,H-2,인터뷰,(해당 없음),08:30,(미기재),H-1 말로는 교량 쪽이 위험하다고 함,H-1,2차 전달,M1·M2(R5)\nC3,I-1,사진자료,09:10,09:30,나-3 일대,차량형 물체 8개 판독(구름 일부),,해상도 보통,M1·M2(R2)·M3(K1-B·K3-B)·M5\nC4,O-1,공개자료,(미확인),10:20,(미확인),교량 앞 군용 행렬 사진 1장,,촬영 시각·장소 미확인·재사용 의심,M1·M2(R3)·M3(K3-A)·M5\nO2-1,O-2,공개자료,(해당 없음),10:45,(미확인),O-1 사진을 인용한 게시물,O-1,재인용,M7 참고\nK4B,H-2,인터뷰,(미기재),10:50,가-4,해당 조직이 정비 중이라고 함,(미기재),전언 여부 미기재,M3(K4-B)\nK2A,I-2,사진자료,11:00,11:05,다-2 도로,이동 차량 없음,,배터리 교체 공백 10:35~10:55 직후 촬영,M3(K2-A)·M5·M8(A)\nK2B,H-3,인터뷰,11:00경,11:40,다-2 도로,차량 행렬 통과,,'경' 표기,M3(K2-B)·M5·M8(A)\n", '02_데이터셋/20261019b_선택미션_대체입력.md': '# [교육용가상자료] 선택 미션 미수행 시 제공 대체입력\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\n제공 기본값은 정답이나 학습자 수행물이 아니다. 후속 미션에서 사용한 버전·값·자료 근거를 표시한다.\n\n## 미션 4 대체 규칙 v1\n\n가중치: 지지 +1, 반대 -1, 미확인 ?의 감점계수 1. 점수=지지합−반대합−?개수. 여기서 반대합은 음수 가중치를 합친 값의 절댓값입니다. 반대 근거와 미확인 항목은 점수를 낮춥니다. 규칙1: 반대 근거 또는 미해소 상충이 하나라도 있으면 보류. 규칙2: 점수3 미만이면 추가 수집 요청. 규칙3: 독립 출처 유형 N은 미션 2에서 정한 값을 그대로 사용(미정이면2). 규칙4: 사람의 최종 판단·운영 규칙 우선, 자동 승인 금지. 미션 4의12항목 설계·S1~S3·레드팀 제출은 요구하지 않는다. 미션 5에서는 독립 출처 개수·상충을 실제 자료로 대조한다.\n\n## 미션 6 대체 빈 서식\n\n| 보고ID | 출처ID | 사실 | 판단 | 조치 | 미확인 | 관측시각 | 접수시각 | 근거 |\n|---|---|---|---|---|---|---|---|---|\n| | | | | | | | | |\n\n미션 9의 요약 차트는 미션 5·미션 7 결과를 이 빈 서식에 정리한다. 미션 6 자체의 보고3종 전사·8출처 대조 수행은 요구하지 않는다.\n\n## 미션 8 대체 기준 유지안 v1\n\n이전 기준=미션 2·미션 5 보존본, 수정 기준=동일 기준 유지안 v1. 원문 AAR A·B를 미션 9에서 읽어 B1~B5와 함께7건에 두 기준을 적용한다. 이전 판단이 없는 A·B는 미션 9에서 이전 기준을 재적용하고 과거 수행값처럼 쓰지 않는다. 같은 결과는 유지 이유를 기록한다. 미션 8 쟁점5건·반박·AI 세션 제출은 요구하지 않는다.\n\n## 미션 10 부록\n\n수행한 선택 미션은 실제 산출물을, 미수행한 선택 미션은 이 제공 기본값·빈 서식과 후속 필수 미션에서 작성한 적용 기록을 부록에 넣는다. 원본은 보존하고 기준 유지·변경 내역을 표시한다.\n', '03_참고자료/20261019b_정오표.md': '# 실습 적용 기준 — 원문 보존 / 수정본 채점\n\n원문은 보존 원문 및 학습자용 보존본에 남아 있습니다. 아래 수정은 원문 사실 변경이나 현재 실습에 적용할 변경 사항입니다.\n\n| 항목 | 원문 값/종전 값 | 수정 값 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 선택 미션 평가 | 선택 미션 Q1만 선택, 산출물·Q2·Q3 필수(이전 안내) | 미션 4·미션 6·미션 8 산출물·Q1·Q2·Q3 모두 가점/참고 | 최종 PASS 범위 충돌 해소 | 필수 7미션의 Q1·Q2·Q3만 최종 PASS |\n| 정확성 척도 | 정확성 ( )단계 이하 | 숫자 코드 n 이상(n~6) | 낮은 신뢰 수준과 숫자 방향 구별 | 미션 2 서식3 및 후속 규칙 |\n| 접수·관측 시각 | 관측 미기재이면 접수 차이 계산 불가 | 접수 차이와 관측 차이 각각 계산 | 서로 다른 시간열 | 미션 3 K1 접수 차이110분, 관측 차이 계산 불가 |\n| 개인 독립 판정 | 조원 1~4 열 / 4건 이상 일치 | 학습자·AI1·AI2·해당 없음 / 보고5건 중4건 | 평가자 수와 보고 건수 구분 | 미션 2 서식2 |\n| 잠정 행정판단 | 이 미션에서는 결론 금지 | 사실 단정 금지, 잠정 행정판단 필수 | 판단 종류 구분 | 미션 3 서식1·확인 방법 |\n| 미션 7 카드 관측시각 | D5 같은 시각, 기준 없음 | D1·D5 관측09:10; C3와 별도 카드 실험 | 수량 상충 판단 근거 제공; 10여 대/3개 유지 | 미션 7 카드 실험; 경과시간 분:초는 별도 |\n| 학습시간 | 원문 소요시간/배정시간 이중 기준 | 학습 계획은 원문 시간 기준 | 원문 합계27시간 유지; 현실 시간 미실측 | 전 미션 시간 계획 |\n| 도메인 예시 | .go.kr 예시에 교육장 제공 자료 | .or.kr 예시로 이동; 기관·발행자 확인 우선 | 도메인 형태와 신뢰도 혼동 방지 | 학습 안내 문서; 접속 확인 아님 |\n', '02_데이터셋/isr06_m08_sample.md': "# ISR06 미션 8 실습 샘플\n\n> `02_데이터셋/ISR06_실습자료_수정실습본.md`의 미션 8 부분입니다. 전체 자료는 프로젝트 첨부 자료의 학습자배포 ZIP에 있습니다.\n\n**미션 8. AAR을 통한 보고 불일치 해결 실패 원인 분석 및 교차검증 루브릭 개선**\n\n가. 제공 자료 1 - 가상 AAR 일지 A(상충 해결 실패 사례) [교육용가상자료]\n\n| **시각** | **기록** |\n| --- | --- |\n| **08:10** | H-1(차량 10여 대)과 I-1(8개) 수량 차이 확인. 상충으로 분류하지 않고 '대체로 일치'로 기록. |\n| **09:00** | 드론 I-2 영상에서 다-2 도로 차량 없음. 담당 분석관이 'H-3은 오보'로 단정. |\n| **09:20** | H-3이 영상 촬영 이후 시각의 보고임이 뒤에 확인됨. 시차 가능성은 검토되지 않음. |\n| **09:40** | 상급 보고서에 '이동 확인'과 '이동 없음'이 한 문서에 함께 기재됨. |\n\n가. 제공 자료 2 - 가상 AAR 일지 B(확증 편향 사례) [교육용가상자료]\n\n| **시각** | **기록** |\n| --- | --- |\n| **10:00** | 분석관이 '교량 통과' 가설을 먼저 세움. 이후 일치하는 보고만 선별해 정리. |\n| **10:30** | 반대 정황(통신량 정상, 영상 차량 3개)이 있었으나 '일시적'이라며 주석 처리. |\n| **11:10** | 동료 분석관이 재검토를 제안했으나 일정 이유로 보류됨. |\n| **13:00** | 가상 후속 보고에서 교량 통과가 없었던 것으로 판명됨. |\n\n나. 작성 서식 1 - AAR 원인 분석표(비난 없이 사실 중심으로 작성)\n\n| **사례** | **무엇이 계획되었나** | **실제로 무슨 일이 있었나** | **원인(사람·절차·정보·통신)** | **개선안** |\n| --- | --- | --- | --- | --- |\n| **A** | | | | |\n| **A** | | | | |\n| **B** | | | | |\n| **B** | | | | |\n\n나. 작성 서식 2 - 레드팀 쟁점·대응표(5건 이상, 예시 1건)\n\n| **번호** | **취약한 전제** | **레드팀 비판** | **작성 팀 반박 또는 수정안** | **경고 문구** |\n| --- | --- | --- | --- | --- |\n| **1** | 신뢰성 A 출처는 틀리지 않는다 | 신뢰성이 높아도 관측 오류·전달 지연이 있다 | 신뢰성과 정확성을 따로 평가하고 정확성은 독립 확인으로 결정 | '신뢰성 A여도 단독 근거로 승인하지 않는다' |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n| **6** | | | | |\n\n나. 작성 서식 3 - 반대 방향 위험 점검: 보류가 지나치게 잦을 때 생기는 위험(정보 지원 지연, 책임자의 불신)과 이를 줄이는 방법(잠정 등급 보고, 시간 목표)을 3줄 이내로 적는다.\n\n나. 작성 서식 4 - 수정 반영 계획표\n\n| **번호** | **수정할 문서·조항** | **수정 내용** | **담당(가상 직책)** | **완료 기한** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| AAR 일지 2건을 시간순으로 재구성하고 계획과 실제의 차이를 적었다 | ( ) |\n| 원인을 사람·절차·정보·통신으로 분류하고 개인 비난 표현이 없다 | ( ) |\n| 레드팀 쟁점을 5건 이상 도출했다 | ( ) |\n| 모든 쟁점에 반박 또는 수정안을 적었다(100%) | ( ) |\n| 취약한 전제마다 경고 문구를 달았다 | ( ) |\n| 보류 과다에 따른 반대 방향 위험을 다뤘다 | ( ) |\n", 'reports/isr06_redteam_critique_report.md': '# 미션 8 reports/isr06_redteam_critique_report.md(AAR 원인 분석표, 레드팀 쟁점·대응표, 검토 회의록, 수정 반영 계획표) (선택·가점/참고)\n\n빈 작성 양식입니다. 실제 수행 후 작성하세요.\n\n## 산출물 본문\n\n(여기에 작성)\n\n## 자료 ID와 판단 근거\n\n(여기에 작성)\n\n## 협업·AI 기록과 선별 이유\n\n(여기에 작성)\n\n## 점검 결과와 미수행 항목\n\n(여기에 작성)\n'}

for name, content in payload.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8')
print(f'미션 8 자료 준비 완료 ({len(payload)}개 파일 확인, 기존 작성 파일 보존)')

### 할 일

1. 두 일지를 각각 시간순으로 정리하고 계획과 실제의 차이를 찾습니다.

2. AI에게 사람·절차·정보·통신으로 원인을 나눈 개선 문서 초안을 요청합니다.

3. 별도 대화에서 취약한 전제 5건 이상을 지적받고 근거와 대조합니다.

4. 반대 방향의 위험과 대응을 적고 지적→바꾼 조항→변화의 3열 표를 만듭니다.

5. 원인 분석표·쟁점 대응표·수정 계획과 검토 기록을 저장합니다.

평가의 자료 수·숫자·근거 조건은 같은 미션 문제지의 확인 포인트와 대조합니다.

### 수행 단계 (guideSteps 요약)
1. **두 일지를 각각 시간순으로 정리하고 계획과 실제의 차이를 찾습니다**
2. **AI와 함께 해보기 — 문제가 생긴 원인과 문서 개선안 초안**
3. **별도 대화에서 취약한 전제 5건 이상을 지적받고 근거와 대조합니다**
4. **반대 방향의 위험과 대응을 적고 지적→바꾼 조항→변화의 3열 표를 만듭니다**
5. **원인 분석표·쟁점 대응표·수정 계획과 검토 기록을 저장합니다**

---

가. 제공 자료 1 - 가상 AAR 일지 A(상충 해결 실패 사례) [교육용가상자료]

| **시각** | **기록** |
| --- | --- |
| **08:10** | H-1(차량 10여 대)과 I-1(8개) 수량 차이 확인. 상충으로 분류하지 않고 '대체로 일치'로 기록. |
| **09:00** | 드론 I-2 영상에서 다-2 도로 차량 없음. 담당 분석관이 'H-3은 오보'로 단정. |
| **09:20** | H-3이 영상 촬영 이후 시각의 보고임이 뒤에 확인됨. 시차 가능성은 검토되지 않음. |
| **09:40** | 상급 보고서에 '이동 확인'과 '이동 없음'이 한 문서에 함께 기재됨. |

가. 제공 자료 2 - 가상 AAR 일지 B(확증 편향 사례) [교육용가상자료]

| **시각** | **기록** |
| --- | --- |
| **10:00** | 분석관이 '교량 통과' 가설을 먼저 세움. 이후 일치하는 보고만 선별해 정리. |
| **10:30** | 반대 정황(통신량 정상, 영상 차량 3개)이 있었으나 '일시적'이라며 주석 처리. |
| **11:10** | 동료 분석관이 재검토를 제안했으나 일정 이유로 보류됨. |
| **13:00** | 가상 후속 보고에서 교량 통과가 없었던 것으로 판명됨. |

### AI와 함께 해보기

교육장에서 제공하는 생성형 AI에 아래 요청을 복사하고 자료를 채웁니다.

```text
너는 가상 조직의 문서 작성 도우미야.
아래 자료만 사용해 「문제가 생긴 원인과 문서 개선안」의 초안을 만들어 줘.
자료: 서로 독립된 업무 일지 A·B, 미션 2·4·5 기준.
[여기에 해당 가상 자료와 내 기준을 붙여넣기]
형식: 시각 | 계획 | 실제 | 근거. 원인 | 조항 번호 | 개선 문장 | 반대 위험. 지적 | 바꾼 줄 | 달라진 결과.
조건: 자료 ID를 붙여 줘. 사실·숫자·시각을 만들지 말고 알 수 없으면 미확인으로 써 줘. 단정 대신 근거와 한계를 함께 적어 줘. 최종 판단은 내가 할게.
```

**확인할 것 3가지**

1. 사실·숫자: 자료 ID, 원문, 수량, 단위, 시각을 자료와 대조합니다.

2. 빠진 항목: 요청한 표의 열과 미확인·근거·책임 항목이 있는지 봅니다.

3. 과장 표현: 근거 없이 반드시·확실하다라고 쓴 문장을 찾습니다.

### 작성 서식 원본 (참고)

나. 작성 서식 1 - AAR 원인 분석표(비난 없이 사실 중심으로 작성)

| **사례** | **무엇이 계획되었나** | **실제로 무슨 일이 있었나** | **원인(사람·절차·정보·통신)** | **개선안** |
| --- | --- | --- | --- | --- |
| **A** | | | | |
| **A** | | | | |
| **B** | | | | |
| **B** | | | | |

나. 작성 서식 2 - 레드팀 쟁점·대응표(5건 이상, 예시 1건)

| **번호** | **취약한 전제** | **레드팀 비판** | **작성 팀 반박 또는 수정안** | **경고 문구** |
| --- | --- | --- | --- | --- |
| **1** | 신뢰성 A 출처는 틀리지 않는다 | 신뢰성이 높아도 관측 오류·전달 지연이 있다 | 신뢰성과 정확성을 따로 평가하고 정확성은 독립 확인으로 결정 | '신뢰성 A여도 단독 근거로 승인하지 않는다' |
| **2** | | | | |
| **3** | | | | |
| **4** | | | | |
| **5** | | | | |
| **6** | | | | |

나. 작성 서식 3 - 반대 방향 위험 점검: 보류가 지나치게 잦을 때 생기는 위험(정보 지원 지연, 책임자의 불신)과 이를 줄이는 방법(잠정 등급 보고, 시간 목표)을 3줄 이내로 적는다.

나. 작성 서식 4 - 수정 반영 계획표

| **번호** | **수정할 문서·조항** | **수정 내용** | **담당(가상 직책)** | **완료 기한** |
| --- | --- | --- | --- | --- |
| **1** | | | | |
| **2** | | | | |
| **3** | | | | |
| **4** | | | | |
| **5** | | | | |

다. 점검표

| **점검 항목** | **확인** |
| --- | --- |
| AAR 일지 2건을 시간순으로 재구성하고 계획과 실제의 차이를 적었다 | ( ) |
| 원인을 사람·절차·정보·통신으로 분류하고 개인 비난 표현이 없다 | ( ) |
| 레드팀 쟁점을 5건 이상 도출했다 | ( ) |
| 모든 쟁점에 반박 또는 수정안을 적었다(100%) | ( ) |
| 취약한 전제마다 경고 문구를 달았다 | ( ) |
| 보류 과다에 따른 반대 방향 위험을 다뤘다 | ( ) |

## [교수자용 해설 및 모범 답안]

### 1. 지도 포인트 (일지 A·B 시각 분리 & 비난 금지)
- 일지 A(`08:10, 09:00, 09:20, 09:40`)와 일지 B(`10:00, 10:30, 11:10, 13:00`)는 별개 사례이므로 접수대장 시각과 합치지 않습니다.
- 원인 열에 개인 실명이나 `'부주의'`, `'태만'`, `'실수'` 표현이 **0건**이어야 합니다.

### 2. 모범 답안 — 작성 서식 1(AAR 원인 분석표) & 3열 인과표
- **사례 A**:
  - `08:10`: (계획) M3 상충 분류 기준 적용 / (실제) `10여 대 vs 8개`를 `'대체로 일치'`로 처리 / (원인: **절차**) 수치 차이 허용 임계 및 상충 분류 절차 미준수 / (개선안) 수량 차이 발생 시 무조건 M3 상충표 등록 의무화
  - `09:00~09:20`: (계획) M3 시차 가능성 점검 / (실제) `I-2` 미관측만으로 `H-3`을 오보 단정 후 시차 확인 / (원인: **정보·절차**) 관측 시각 대조 전 단정 금지 규칙 부재 / (개선안) 반대 보고 간 시각 차이 검증 완료 전 오보 단정 금지
- **사례 B**:
  - `10:00~10:30`: (계획) M4/M5 분기 3 반대 근거 우선 점검 / (실제) `'교량 통과'` 가설 선설정 후 반대 정황 주석 처리 / (원인: **사람·절차**) 가설 선행 확증 편향을 막는 체크리스트 부재 / (개선안) 반대 근거 1건 이상 존재 시 자동 보류(분기 3) 강제
  - `11:10~13:00`: (계획) 동료 재검토 요청 수용 / (실제) 일정 이유로 재검토 보류 후 오판 / (원인: **통신·절차**) 재검토 요청 에스컬레이션 절차 미비 / (개선안) 동료 이견 제기 시 보류 상태 전환 의무화
- **반대 방향 위험(서식 3)**: 과도한 보류 시 정보 지원 지연·책임자 불신이 발생하므로, 잠정 등급·미확인 목록·재확인 목표 시각을 함께 보고해 의사결정 지연을 방지함.

In [ ]:
# [교수자용 검증 코드] M8 AAR 원인 문장 비난 표현 0건 검증
forbidden = ["부주의", "태만", "실수", "게으름", "무능"]
model_causes = [
    "절차: 수량 차이 상충 분류 임계 기준 미비",
    "정보·절차: 관측 시각 대조 전 오보 단정 방지 규칙 부재",
    "사람·절차: 가설 선행 확증 편향 차단 체크리스트 부재",
    "통신·절차: 동료 재검토 이견 제기 시 보류 전환 절차 미비",
]
hits = [w for c in model_causes for w in forbidden if w in c]
assert len(hits) == 0
print("[M8 검증 PASS] 모범 원인 분석 4건 비난 표현 0건 확인")